![JohnSnowLabs](https://sparknlp.org/assets/images/logo.png)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JohnSnowLabs/spark-nlp/blob/master/examples/python/transformers/onnx/HuggingFace_ONNX_in_Spark_NLP_CrossEncoder.ipynb)

## Import ONNX CrossEncoder models from HuggingFace 🤗 into Spark NLP 🚀

Let's keep in mind a few things before we start 😊

- ONNX support was introduced in `Spark NLP 5.0.0`, enabling high performance inference for models.
- `CrossEncoder` is a pairwise BERT sequence classifier: it jointly encodes a `(query, passage)` pair as `[CLS] query [SEP] passage [SEP]` and returns a single relevance score in `[0, 1]` (sigmoid). It is the Spark NLP equivalent of a `sentence-transformers` `CrossEncoder`.
- This annotator supports the ONNX engine only.
- You can import any BERT-family cross-encoder with a single-logit regression head, e.g. `cross-encoder/ms-marco-MiniLM-L6-v2`.

## Export and Save HuggingFace model

- Let's install `transformers` and `optimum` to export the model to ONNX.
- We lock `transformers` to a known good version for reproducibility.

In [ ]:
!pip install -q --upgrade transformers[onnx]==4.51.3 optimum onnx

- HuggingFace's `optimum` exports a standard `AutoModelForSequenceClassification` checkpoint to ONNX directly.
- `cross-encoder/ms-marco-MiniLM-L6-v2` has `num_labels=1` (a regression head), so no custom tracing is needed.

In [ ]:
from optimum.onnxruntime import ORTModelForSequenceClassification
from transformers import AutoTokenizer

MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L6-v2"
EXPORT_PATH = f"onnx_models/{MODEL_NAME}"

ort_model = ORTModelForSequenceClassification.from_pretrained(MODEL_NAME, export=True)
ort_model.save_pretrained(EXPORT_PATH)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.save_pretrained(EXPORT_PATH)

Let's have a look inside the export directory:

In [ ]:
!ls -l {EXPORT_PATH}

- We need to move `vocab.txt` into an `assets` folder, which is where Spark NLP looks for tokenizer assets.
- The regression head has no labels, so no `labels.txt` is required.

In [ ]:
!mkdir -p {EXPORT_PATH}/assets
!mv {EXPORT_PATH}/vocab.txt {EXPORT_PATH}/assets/

In [ ]:
!ls -lR {EXPORT_PATH}

Voila! We have our `vocab.txt` inside the `assets` folder and the `model.onnx` at the export root.

## Import and Save CrossEncoder in Spark NLP

- Let's install and set up Spark NLP in your environment. Make sure the version you install includes `CrossEncoder`.

In [ ]:
!pip install -q spark-nlp pyspark==3.5.4

Let's start a Spark NLP session:

In [ ]:
import sparknlp

spark = sparknlp.start()
print(sparknlp.version())

- Let's use `loadSavedModel`, which takes the export folder and a `SparkSession`.
- The two input columns are the two `DOCUMENT` columns of the pair (query and passage).

In [ ]:
from sparknlp.annotator import CrossEncoder

crossEncoder = CrossEncoder.loadSavedModel(EXPORT_PATH, spark) \
    .setInputCols(["document1", "document2"]) \
    .setOutputCol("score") \
    .setCaseSensitive(False)

- Let's save it on disk so it is easier to be moved around and reused later.

In [ ]:
crossEncoder.write().overwrite().save(f"{MODEL_NAME}_spark_nlp_onnx")

Let's clean up the export folder we no longer need:

In [ ]:
!rm -rf {EXPORT_PATH}

Now let's load the saved Spark NLP model back:

In [ ]:
crossEncoder_loaded = CrossEncoder.load(f"{MODEL_NAME}_spark_nlp_onnx") \
    .setInputCols(["document1", "document2"]) \
    .setOutputCol("score")

This is how you use it for reranking. A `MultiDocumentAssembler` builds the two `DOCUMENT` columns, one query duplicated against several passages (a layout you build upstream with a `crossJoin`/`explode`).

In [ ]:
from sparknlp.base import MultiDocumentAssembler
from pyspark.ml import Pipeline

document = MultiDocumentAssembler() \
    .setInputCols(["query", "passage"]) \
    .setOutputCols(["document1", "document2"])

pipeline = Pipeline(stages=[document, crossEncoder_loaded])

query = "How many people live in Berlin?"
passages = [
    "Berlin has a population of 3,520,031 registered inhabitants in an area of 891.82 square kilometers.",
    "Berlin is well known for its museums.",
    "In 2014, the city state Berlin had 37,368 live births.",
]

data = spark.createDataFrame([[query, p] for p in passages]).toDF("query", "passage")
result = pipeline.fit(data).transform(data)
result.select("passage", "score.result").show(truncate=80)

That's it! You can now go wild and use hundreds of `CrossEncoder` models as rerankers in Spark NLP 🚀